<a href="https://colab.research.google.com/github/jlloring/ST-554_JLoring/blob/main/Loring_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ST 554 Homework 7**
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

In [28]:
# importing required modules
import pandas as pd
import numpy as np
from sklearn import linear_model
from math import sqrt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, Ridge, ElasticNet, \
                                 LogisticRegressionCV, LassoCV, RidgeCV, ElasticNetCV
from sklearn.preprocessing import PolynomialFeatures

## **Data Work**
The code below reads in the `winequality-red.csv` and `winequality-white.csv` files from the [UCI machine learning repository site](https://archive.ics.uci.edu/dataset/186/wine+quality). These files must be re-uploaded to the *Files* section at the start of each new session!

In [29]:
white = pd.read_csv("winequality-white.csv", sep = ';')
red = pd.read_csv("winequality-red.csv", sep = ';')

Now, we create a variable called `type` within each dataset that represents the type of wine (i.e., red or white). Since we know this binary identification will be used for logistic regression, we will also create a variable called `type_num` that takes on a value of 1 for red wine and 0 for white wine.

In [30]:
white['type'] = 'White'
white['type_num'] = 0
red['type'] = 'Red'
red['type_num'] = 1

To verify that our data is looking as expected, we use the `.head()` method on each dataset.

In [31]:
# check white wine data
white.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_num
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6,White,0
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6,White,0
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6,White,0
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,White,0
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,White,0


In [32]:
#check red wine data
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_num
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,Red,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,Red,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,Red,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,Red,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,Red,1


Now, we use the `concat()` function from `pandas` to combine these two datasets.

In [33]:
wine = pd.concat([white, red])
wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_num
0,7.0,0.270,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6,White,0
1,6.3,0.300,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6,White,0
2,8.1,0.280,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6,White,0
3,7.2,0.230,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6,White,0
4,7.2,0.230,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6,White,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5,Red,1
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6,Red,1
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6,Red,1
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5,Red,1


We now have our final working dataset, `wine`. According to the UCI machine learning repository, there are no missing values, so we do not need to do any additional data cleaning!

## **Splitting the Data**
The code below splits up the `wine` data into a training and test set. Stratified sampling is used to make sure that we have a similar proportion of white and red wines in the training and test sets. The `train_test_split()` function from `sklearn.model_selection` will be used. This splitting is done two separate times for our two separate tasks:
1. **Regression Task** (`alcohol` as Response)
2. **Classification Task** (`type_num` as Response)

#### **Regression Task Splitting**
The code below splits the `wine` dataset into training and testing data for use with the Regression Task.

In [34]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
wine.drop(['alcohol', 'type'], axis = 1), #drops the type category text, keeps binary version
wine['alcohol'],
stratify = wine['type'], #stratifies on wine type
test_size = 0.20,
random_state = 44)

##### **Regression Task Standardizing**
Now that we have our training and testing data, we want to standardize our values! The code below accomplishes this.

In [35]:
means_reg = X_train1.apply(np.mean, axis = 0) #grabs means of all training data columns
stds_reg = X_train1.apply(np.std, axis = 0) #grabs standard deviations of all training data columns

#overwrites training data with standardized values
X_train1 = X_train1.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)

*Note: The standardizations applied to the training data above will be used on the test data as well!*

#### **Classification Task Splitting**

In [36]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
wine.drop(['type_num', 'type'], axis = 1), #drops the type category text, keeps binary version
wine['type_num'],
stratify = wine['type'], #stratifies on wine type
test_size = 0.20,
random_state = 44)

Now that our data is split for each task, we can start making our models!

## **Regression Task: `alcohol` as Response**

In this section, we use a variety of regression techniques to fit both training models and tests models. We will use:
- Multiple Linear Regression (MLR)
- LASSO
- Ridge Regression
- Elastic Net

### **Train Models**

In this sub-section, we fit the four types of models described above on our training data.

#### **Multiple Linear Regression**

Here are the four different MLR models that I will choose to fit (listed predictors):
1. `citric acid`, `residual sugar`, `pH`
2. `volatile acidity`, `residual sugar`, `total sulfur dioxide`, `density`, `type_num`
3. `fixed acidity`, `pH`, and their interaction
4. `residual sugar`, `density`, `residual sugar`$^{2}$, `density`$^{2}$, the interaction of density and residual sugar

##### **MLR #1**
The code below fits the first model described at the beginning of the MLR section.

In [37]:
MLR_1 = linear_model.LinearRegression().fit(X_train1[['citric acid', 'residual sugar', 'pH']],
                                            y_train1)
print(MLR_1.intercept_, MLR_1.coef_)

10.467236867423512 [ 0.06027543 -0.43050488  0.03963814]


The order of the coefficients corresponds to the order of the columns in the `wine` dataframe. Therefore, the correct order is `citric acid`, `residual sugar`, and `pH`, respectively.

##### **MLR #2**
The code below fits the second model described at the beginning of the MLR section.

In [38]:
MLR_2 = linear_model.LinearRegression().fit(X_train1[['volatile acidity', 'residual sugar',
                                                      'total sulfur dioxide', 'density',
                                                      'type_num']],
                                            y_train1)
print(MLR_2.intercept_, MLR_2.coef_)

10.467236867423505 [-0.03002403  0.66125381 -0.11014287 -1.45206785  0.72135357]


The order of the coefficients corresponds to the order of the columns in the `wine` dataframe. Therefore, the correct order is `volatile acidity`, `residual sugar`, `total sulfur dioxide`, `density`, and `type_num`, respectively.

##### **MLR #3**
The code below fits the third model described at the beginning of the MLR section. This requires multiple steps since we are using an interaction term in this model.

In [39]:
#creates interaction object
poly3 = PolynomialFeatures(interaction_only=True, include_bias = False)

#fits interaction object with variables we want
design3 = poly3.fit_transform(X_train1[['fixed acidity', 'pH']])

#fits MLR model with interactions
MLR_3 = linear_model.LinearRegression().fit(X = design3, y = y_train1)

print(MLR_3.intercept_, MLR_3.coef_)

10.433094946500994 [-0.10007542  0.09546757 -0.13761727]


We can use the `.get_feature_names_out()` method to confirm the order of the coefficients since we are using `PolynomialFeatures`.

In [40]:
poly3.get_feature_names_out(['fixed acidity', 'pH'])

array(['fixed acidity', 'pH', 'fixed acidity pH'], dtype=object)

Therefore, the order of the coefficients is `fixed acidity`, `pH`, and their interaction, respectively.

##### **MLR #4**
The code below fits the fourth and final model described at the beginning of the MLR section. This also requires multiple steps since we are using both quadratic terms and a interaction term in this model.

In [41]:
#creates interaction & polynomial object
poly4 = PolynomialFeatures(degree = 2, interaction_only = False, include_bias = False)

#fits interaction object with variables we want
design4 = poly4.fit_transform(X_train1[['residual sugar', 'density']])

#fits MLR model with interactions
MLR_4 = linear_model.LinearRegression().fit(X = design4, y = y_train1)

print(MLR_4.intercept_, MLR_4.coef_)

10.112001788632426 [ 0.22766094 -1.01870269  0.26248177 -0.68736313  0.46672017]


Again, we can use the `.get_feature_names_out()` method to confirm the order of the coefficients since we are using `PolynomialFeatures`.

In [42]:
poly4.get_feature_names_out(['residual sugar', 'density'])

array(['residual sugar', 'density', 'residual sugar^2',
       'residual sugar density', 'density^2'], dtype=object)

Therefore, the order of the coefficients is `residual sugar`, `density`, `residual sugar`$^{2}$, the interaction between residual sugar and density, and `density`$^{2}$, respectively.

##### **CV with MLR Models**
The code below uses cross-validation with each of the above MLR models to help select the best one.

In [43]:
#cross-validation on MLR #1
cv1_MLR = cross_validate(linear_model.LinearRegression(),
                         X_train1[['citric acid', 'residual sugar', 'pH']],
                         y_train1,
                         cv = 5,
                         scoring = 'neg_mean_squared_error',
                         return_train_score = True)

#cross-validation on MLR #2
cv2_MLR = cross_validate(linear_model.LinearRegression(),
                         X_train1[['volatile acidity', 'residual sugar', 'total sulfur dioxide',
                                   'density', 'type_num']],
                         y_train1,
                         cv = 5,
                         scoring = 'neg_mean_squared_error',
                         return_train_score = True)

#cross-validation on MLR #3
cv3_MLR = cross_validate(linear_model.LinearRegression(),
                         design3,
                         y_train1,
                         cv = 5,
                         scoring = 'neg_mean_squared_error',
                         return_train_score = True)

#cross-validation on MLR #4
cv4_MLR = cross_validate(linear_model.LinearRegression(),
                         design4,
                         y_train1,
                         cv = 5,
                         scoring = 'neg_mean_squared_error',
                         return_train_score = True)

Now that cross-validation is complete, we can compare the CV RMSE values to select the best model. Lower is better!

In [44]:
print(
    round(np.sqrt(-sum(cv1_MLR["test_score"])/5),4),
    round(np.sqrt(-sum(cv2_MLR["test_score"])/5),4),
    round(np.sqrt(-sum(cv3_MLR["test_score"])/5),4),
    round(np.sqrt(-sum(cv4_MLR["test_score"])/5),4)
    )

1.098 0.6905 1.1638 0.7272


Since the RMSE is lowest for the second MLR model, that is the best choice of the four fitted models!

In [45]:
MLR_best = MLR_2
print(MLR_best.intercept_)
print(np.array(list(zip(X_train1[['volatile acidity', 'residual sugar',
                                   'total sulfur dioxide', 'density',
                                   'type_num']], MLR_best.coef_))))

10.467236867423505
[['volatile acidity' '-0.03002402751932421']
 ['residual sugar' '0.6612538082282409']
 ['total sulfur dioxide' '-0.11014286805865692']
 ['density' '-1.4520678529538855']
 ['type_num' '0.7213535715000869']]
